# Introductory Statistics – Part 3: Sampling and Statistical Inference

Welcome to Part 3. We now bridge the gap between probability and real-world data analysis by introducing **sampling theory** and the foundations of **statistical inference**.

**By the end of this notebook you will be able to:**

- Explain the Central Limit Theorem and demonstrate it via simulation
- Apply the normal approximation to the binomial distribution
- Construct and interpret **confidence intervals** for proportions and means
- Calculate required sample sizes for a desired margin of error
- Conduct a **one-sample z-test** and state conclusions in context

**Topics covered:**

9. Simple random sampling, sampling distributions, CLT, normal approximation to the binomial  
10. Confidence intervals for proportions, margin of error, and sample size  
11. Confidence interval for a population mean and sample size  
12. Hypothesis testing foundations

> **Prerequisite:** Part 2 – Probability & Distributions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

sns.set(style="whitegrid")
np.random.seed(42)

---
## 9. Sampling, Sampling Distributions, and the Central Limit Theorem

### 9.1 Simple Random Sampling

A **simple random sample (SRS)** gives every member of a population an equal chance of selection. SRS is the gold standard because it avoids systematic bias.

### 9.2 Sampling Distribution of $\bar{x}$

If we repeatedly draw samples of size $n$ from a population with mean $\mu$ and standard deviation $\sigma$, the distribution of sample means $\bar{x}$ has:

| Property | Value |
|---|---|
| Mean | $\mu_{\bar{x}} = \mu$ |
| Standard error | $\sigma_{\bar{x}} = \dfrac{\sigma}{\sqrt{n}}$ |

### 9.3 Central Limit Theorem (CLT)

> For a **large enough** sample size $n$, the sampling distribution of $\bar{x}$ is approximately $N\!\left(\mu,\, \dfrac{\sigma^2}{n}\right)$, **regardless of the population's shape**.

A common rule of thumb: $n \geq 30$ is usually sufficient.

In [ ]:
# Demonstrating the CLT: population is right-skewed (exponential)
# We repeatedly sample from it and observe that the distribution of
# sample means becomes approximately normal.

population = np.random.exponential(scale=10, size=200_000)
pop_mean   = population.mean()
pop_std    = population.std()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot the skewed population
axes[0].hist(population, bins=80, density=True, color="coral", edgecolor="none")
axes[0].set_title("Population (Exponential)")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Density")

# CLT in action for different sample sizes
for ax, n, color in zip(axes[1:], [5, 50], ["steelblue", "teal"]):
    sample_means = [np.random.choice(population, size=n, replace=False).mean()
                    for _ in range(3000)]
    ax.hist(sample_means, bins=40, density=True, color=color,
            edgecolor="none", alpha=0.8, label="Sample means")

    # Overlay theoretical normal
    se = pop_std / np.sqrt(n)
    x_range = np.linspace(min(sample_means), max(sample_means), 300)
    ax.plot(x_range, norm.pdf(x_range, pop_mean, se),
            color="black", linewidth=2, linestyle="--", label=f"$N(\\mu, \\sigma^2/{n})$")

    ax.set_title(f"Sample Means ($n = {n}$)")
    ax.set_xlabel("$\\bar{{x}}$")
    ax.legend(fontsize=8)

plt.suptitle("Central Limit Theorem Demonstration", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Population mean (μ): {pop_mean:.2f}")
print(f"Population std dev (σ): {pop_std:.2f}")

### 9.4 Normal Approximation to the Binomial

When $n$ is large, the binomial distribution can be approximated by a normal distribution:

$$X \sim \text{Binomial}(n, p) \approx N\!\left(np,\; np(1-p)\right)$$

**Rule of thumb:** use the approximation only when $np \geq 10$ and $n(1-p) \geq 10$.

In [ ]:
# Compare Binomial(n=100, p=0.4) to its normal approximation
from scipy.stats import binom as binom_dist

n, p = 100, 0.4
x = np.arange(0, n + 1)
binom_pmf = binom_dist.pmf(x, n, p)

mu_b    = n * p
sigma_b = np.sqrt(n * p * (1 - p))
normal_pdf = norm.pdf(x, mu_b, sigma_b)

# Check rule of thumb
print(f"np = {n*p:.0f}  (need ≥ 10)")
print(f"n(1-p) = {n*(1-p):.0f}  (need ≥ 10)")
print(f"Approximation is valid.\n")
print(f"Binomial: μ = {mu_b:.1f},  σ = {sigma_b:.2f}")

plt.figure(figsize=(7, 4))
plt.bar(x, binom_pmf, width=0.8, alpha=0.6, label="Binomial PMF", color="steelblue")
plt.plot(x, normal_pdf, color="red", linewidth=2,
         label=f"Normal approx $N({mu_b:.0f}, {sigma_b:.2f}^2)$")
plt.title(f"Binomial vs Normal Approximation ($n={n},\\, p={p}$)")
plt.xlabel("$k$")
plt.ylabel("Probability")
plt.legend()
plt.tight_layout()
plt.show()

---
## 10. Confidence Intervals for Proportions, Margin of Error, and Sample Size

A **confidence interval (CI)** is a range of plausible values for a population parameter, computed from sample data.

A **95% CI** means: if we repeated the sampling procedure many times, about 95% of the computed intervals would contain the true parameter.

### CI for a population proportion $p$

$$\hat{p} \pm z^* \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

where $z^* = 1.96$ for 95% confidence.

### Margin of error

$$ME = z^* \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

### Required sample size for a desired $ME$

$$n = \frac{z^{*2}\, p(1-p)}{ME^2}$$

Use $p = 0.5$ as a conservative (worst-case) estimate when $p$ is unknown.

In [ ]:
# 95% confidence interval for a population proportion
n          = 200
successes  = 54
phat       = successes / n
z_star     = norm.ppf(0.975)   # 1.96 for 95% CI

se         = np.sqrt(phat * (1 - phat) / n)
lower      = phat - z_star * se
upper      = phat + z_star * se
me         = z_star * se

print(f"Sample proportion (p̂): {phat:.3f}")
print(f"Standard error (SE):   {se:.4f}")
print(f"Margin of error (ME):  {me:.4f}")
print(f"95% CI: ({lower:.3f}, {upper:.3f})")

In [ ]:
# Required sample size for ME = 0.03 at 95% confidence
# Using p = 0.5 (worst case — maximises the required n)

ME_target  = 0.03
p_cons     = 0.5
z_star     = norm.ppf(0.975)

n_required = (z_star**2 * p_cons * (1 - p_cons)) / ME_target**2

print(f"To achieve ME ≤ {ME_target} at 95% confidence:")
print(f"  Required sample size n = {np.ceil(n_required):.0f}")

---
## 11. Confidence Interval for a Population Mean

When $\sigma$ is **known**, the CI for $\mu$ is:

$$\bar{x} \pm z^* \frac{\sigma}{\sqrt{n}}$$

When $\sigma$ is **unknown** (the usual case), we substitute the sample standard deviation $s$ and use the $t$-distribution instead — covered in Part 4.

### Required sample size for a desired CI width $W$

The half-width (margin of error) is $ME = W/2$, so:

$$n = \left(\frac{z^* \sigma}{ME}\right)^2$$

In [ ]:
# 95% CI for a population mean with known σ
np.random.seed(42)
data  = np.random.normal(loc=100, scale=15, size=40)
xbar  = data.mean()
sigma = 15          # assumed known
n     = len(data)
z_star = norm.ppf(0.975)

se    = sigma / np.sqrt(n)
lower = xbar - z_star * se
upper = xbar + z_star * se

print(f"Sample mean (x̄):    {xbar:.2f}")
print(f"Standard error (SE): {se:.3f}")
print(f"95% CI: ({lower:.2f}, {upper:.2f})")
print(f"CI width: {upper - lower:.2f}")

In [ ]:
# Required sample size for desired CI width = 6 (ME = 3), σ = 12, 95% confidence
sigma_plan = 12
ME_plan    = 3       # half the desired CI width
z_star     = norm.ppf(0.975)

n_required = (z_star * sigma_plan / ME_plan) ** 2

print(f"To achieve a CI width of {2*ME_plan} (ME = {ME_plan}):")
print(f"  Required sample size n = {np.ceil(n_required):.0f}")

---
## 12. Hypothesis Testing Foundations

A **hypothesis test** uses sample data to evaluate a claim about a population parameter.

### Structure of a hypothesis test

| Step | Description |
|---|---|
| 1. State hypotheses | $H_0$: null hypothesis (e.g., $\mu = 100$); $H_1$: alternative (e.g., $\mu > 100$) |
| 2. Choose $\alpha$ | Significance level, commonly 0.05 |
| 3. Compute test statistic | e.g., $z = \dfrac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$ |
| 4. Find p-value | Probability of data this extreme if $H_0$ is true |
| 5. Decide | Reject $H_0$ if $p < \alpha$ |

### Error types

| | $H_0$ true | $H_0$ false |
|---|---|---|
| **Reject $H_0$** | Type I error ($\alpha$) | Correct (Power = $1 - \beta$) |
| **Fail to reject $H_0$** | Correct | Type II error ($\beta$) |

In [ ]:
# One-sample z-test for a population mean (σ known)
# H0: μ = 100   (no change)
# H1: μ > 100   (one-sided, we believe the mean is higher)

mu0   = 100
sigma = 10
alpha = 0.05

np.random.seed(7)
data  = np.random.normal(loc=103, scale=10, size=40)
xbar  = data.mean()
n     = len(data)

z_stat  = (xbar - mu0) / (sigma / np.sqrt(n))
p_value = 1 - norm.cdf(z_stat)    # one-sided (right tail)
z_crit  = norm.ppf(1 - alpha)     # critical value

print(f"Sample mean (x̄): {xbar:.3f}")
print(f"Test statistic z: {z_stat:.3f}")
print(f"Critical value z*: {z_crit:.3f}")
print(f"p-value: {p_value:.4f}")
print()
if p_value < alpha:
    print(f"p = {p_value:.4f} < α = {alpha} → Reject H₀.")
    print("Conclusion: sufficient evidence that μ > 100.")
else:
    print(f"p = {p_value:.4f} ≥ α = {alpha} → Fail to reject H₀.")

In [ ]:
# Visualise the test: shade the rejection region and mark the observed z

x_range = np.linspace(-4, 4, 400)
y_range = norm.pdf(x_range)

plt.figure(figsize=(8, 4))
plt.plot(x_range, y_range, color="black", linewidth=2)

# Rejection region (right tail)
plt.fill_between(x_range, y_range, where=(x_range >= z_crit),
                 color="red", alpha=0.4, label=f"Rejection region ($\\alpha={alpha}$)")

# Observed z
plt.axvline(z_stat, color="blue", linestyle="--",
            label=f"Observed $z = {z_stat:.2f}$")
plt.axvline(z_crit, color="red", linestyle="--",
            label=f"Critical value $z^* = {z_crit:.2f}$")

plt.title("One-Sample z-Test: $H_0: \\mu = 100$ vs $H_1: \\mu > 100$")
plt.xlabel("$z$")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

- A **simple random sample** gives every unit an equal chance of selection, minimising bias.
- The **sampling distribution** of $\bar{x}$ has mean $\mu$ and standard error $\sigma/\sqrt{n}$.
- The **CLT** guarantees approximate normality of $\bar{x}$ for $n \geq 30$, even for skewed populations.
- The **normal approximation** to the binomial requires $np \geq 10$ and $n(1-p) \geq 10$.
- A **95% CI** for a proportion: $\hat{p} \pm 1.96\sqrt{\hat{p}(1-\hat{p})/n}$.
- A **95% CI** for a mean (known $\sigma$): $\bar{x} \pm 1.96(\sigma/\sqrt{n})$.
- Hypothesis testing: state $H_0$ and $H_1$, compute $z$, compare $p$-value to $\alpha$.
- **Type I error** = falsely rejecting $H_0$; **Type II error** = failing to reject a false $H_0$.

**Next:** Part 4 – Power, p-values, and the t-Distribution.